# Lennard-Jones флюид и уравнение Ван-дер-Ваальса

Цель ноутбука: численно построить несколько изотерм LJ-флюида, сохранить таблицы EOS, подобрать параметры уравнения Ван-дер-Ваальса и сравнить модель с MD-данными.

In [ ]:
from pathlib import Path
import json

from lj_utils import (
    EOS_FIELDS,
    PROFILE_FIELDS,
    fit_vdw,
    plot_eos_isotherms,
    plot_vdw_fit,
    run_eos_point,
    write_csv,
)


## PARAMS

Все параметры расчёта задаются здесь. Чтобы изменить расчёт, отредактируйте эту ячейку и перезапустите ноутбук.

In [ ]:
PARAMS = {
    "N": 128,
    "sigma": 1.0,
    "epsilon": 1.0,
    "mass": 1.0,
    "rcut": 2.5,
    "dt": 0.002,
    "equil_steps": 500,
    "prod_steps": 1000,
    "sample_interval": 250,
    "friction": 0.2,
    "temperatures": [0.70, 0.90],
    "densities": [0.02, 0.06],
    "seeds": [1],
    "profile_bins": 40,
}

DATA_DIR = Path("data")
FIGURES_DIR = Path("figures")
DATA_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)
PARAMS


## Тестовая EOS-точка

Короткий прогон проверяет, что OpenMM запускается и температура остаётся конечной.

In [ ]:
test_point, test_profile = run_eos_point(
    PARAMS,
    temperature=PARAMS["temperatures"][0],
    density=PARAMS["densities"][0],
    seed=PARAMS["seeds"][0],
    run_id="test",
)
test_point


## Расчёт EOS-сетки

In [ ]:
points = []
profiles = []
run_index = 0

for T in PARAMS["temperatures"]:
    for rho in PARAMS["densities"]:
        for seed in PARAMS["seeds"]:
            run_index += 1
            run_id = f"eos_{run_index:04d}"
            print(f"run {run_id}: T={T}, rho={rho}, seed={seed}")
            point, profile = run_eos_point(PARAMS, T, rho, seed, run_id)
            points.append(point)
            profiles.extend(profile)

write_csv(DATA_DIR / "eos_points.csv", EOS_FIELDS, points)
write_csv(DATA_DIR / "eos_final_profiles.csv", PROFILE_FIELDS, profiles)
len(points), len(profiles)


## Fit Ван-дер-Ваальса и графики

In [ ]:
fit = fit_vdw(points)
(DATA_DIR / "vdw_fit.json").write_text(json.dumps(fit, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")

plot_eos_isotherms(points, FIGURES_DIR / "eos_isotherms.png")
plot_vdw_fit(points, fit, FIGURES_DIR / "vdw_fit.png")
fit


## Результаты

Основные файлы после запуска:

- `data/eos_points.csv`
- `data/eos_final_profiles.csv`
- `data/vdw_fit.json`
- `figures/eos_isotherms.png`
- `figures/vdw_fit.png`
